In [3]:
from sqlalchemy import create_engine
import getpass

user = "root"
password = getpass.getpass("Enter MySQL password: ")
host = "localhost"
db = "sakila"

connection_string = f"mysql+pymysql://{user}:{password}@{host}/{db}"
engine = create_engine(connection_string)

# Test the connection
with engine.connect() as conn:
    print("Connected successfully!")


Enter MySQL password:  ········


Connected successfully!


In [4]:
def rentals_month(engine, month, year):
    
    query = f"""
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = {month}
      AND YEAR(rental_date) = {year};
    """
    
    rentals_df = pd.read_sql(query, engine)
    
    return rentals_df

In [ ]:
may_2005 = rentals_month(engine, 5, 2005)
may_2005.head()

In [5]:
def rental_count_month(df, month, year):
    
    column_name = f"rentals_{month:02d}_{year}"
    
    rental_count = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=column_name)
    )
    
    return rental_count

In [7]:
def compare_rentals(df1, df2):
    
    comparison = pd.merge(
        df1,
        df2,
        on="customer_id",
        how="outer"
    )
    
    comparison = comparison.fillna(0)
    
    rental_columns = comparison.columns[1:]
    
    comparison["difference"] = (
        comparison[rental_columns[1]] -
        comparison[rental_columns[0]]
    )
    
    return comparison